# Import SSB municipality population

Downloads municipality population from Statistics Norway's Statbank API table 06913 and writes a small Parquet dimension table.

The join key is `kommunenummer`, which is already present in the company register as `forretningsadresse.kommunenummer`. 

Source: SSB table 06913, Population and population changes, by region, contents and year. The selected year and extraction timestamp are stored with the output.

In [1]:
import itertools
import json
import os
from datetime import datetime, timezone

import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from staged_write import write_staged

DATA_DIR = "/home/jovyan/data"
SSB_TABLE = "06913"
POPULATION_YEAR = "2026"
OUTPUT_PATH = os.path.join(
    DATA_DIR,
    "parquet",
    f"ssb_population_{POPULATION_YEAR}.parquet",
)
SSB_URL = f"https://data.ssb.no/api/v0/en/table/{SSB_TABLE}"

spark = SparkSession.builder.appName("group13_import_ssb_population").getOrCreate()
print("Spark          ", spark.version)
print("SSB table      ", SSB_TABLE)
print("Population year", POPULATION_YEAR)
print("Output         ", OUTPUT_PATH)

Spark           4.2.0
SSB table       06913
Population year 2026
Output          /home/jovyan/data/parquet/ssb_population_2026.parquet


## Select current municipalities

SSB's region dimension includes counties, the whole country, historical municipality codes, and special regions. Current Norwegian municipality codes are four digits and do not have a suffix such as `u`, which is what the metadata filter below selects.

That filter is not sufficient on its own. For a four-digit code that was not in use in the selected year SSB returns a population of `0` rather than a null, so discarding nulls alone leaves every retired municipality in the table and the extract carries far more codes than Norway has municipalities. A zero here is a retired code, not a measurement, so those rows are dropped and the number dropped is counted rather than assumed.

The distinction matters downstream. `Build_analytics.ipynb` joins this table onto the register left on `kommunenummer`, and `Analyse_geography.ipynb` bands the result by testing for null first. A retired code left in the dimension would therefore join to a population of 0 and be banded as the smallest class of municipality rather than as unknown. Dropping it here makes the join miss instead, which is the honest outcome.

In [2]:
metadata_response = requests.get(SSB_URL, timeout=60)
metadata_response.raise_for_status()
table_metadata = metadata_response.json()

region_variable = next(
    variable for variable in table_metadata["variables"]
    if variable["code"] == "Region"
)

region_codes = [
    code
    for code in region_variable["values"]
    if len(code) == 4 and code.isdigit()
]

assert region_codes, "SSB metadata returned no four-digit municipality candidates"
print("Municipality candidates in metadata:", len(region_codes))

Municipality candidates in metadata: 1184


In [3]:
query = {
    "query": [
        {
            "code": "Region",
            "selection": {
                "filter": "item",
                "values": region_codes,
            },
        },
        {
            "code": "ContentsCode",
            "selection": {
                "filter": "item",
                "values": ["Folkemengde"],
            },
        },
        {
            "code": "Tid",
            "selection": {
                "filter": "item",
                "values": [POPULATION_YEAR],
            },
        },
    ],
    "response": {
        "format": "json-stat2",
    },
}

data_response = requests.post(SSB_URL, json=query, timeout=60)
data_response.raise_for_status()
ssb_payload = data_response.json()
print("Returned dimensions:", ssb_payload["id"])
print("Returned values:", len(ssb_payload["value"]))

Returned dimensions: ['Region', 'ContentsCode', 'Tid']
Returned values: 1184


## Convert JSON-stat2 to rows

JSON-stat2 stores dimension labels and observations separately. The conversion keeps the municipality code as a string so leading zeroes, such as `0301` for Oslo, are preserved.

In [4]:
def ordered_category_codes(dimension):
    category = dimension["category"]
    index = category["index"]
    if isinstance(index, dict):
        return [code for code, _ in sorted(index.items(), key=lambda item: item[1])]
    return list(index)


dimension_order = ssb_payload["id"]
dimension_codes = [
    ordered_category_codes(ssb_payload["dimension"][dimension_name])
    for dimension_name in dimension_order
]

observations = []
for combination, value in zip(
    itertools.product(*dimension_codes),
    ssb_payload["value"],
):
    row = dict(zip(dimension_order, combination))
    row["population"] = value
    observations.append(row)

# SSB reports a population of 0 for a four-digit code that was not in use in the
# selected year, so filtering nulls alone keeps every retired municipality. Both
# classes are counted: the size of the retired set is the reason this filter
# exists, so it is measured here rather than stated.
retired_codes = sorted(
    row["Region"].zfill(4)
    for row in observations
    if row["population"] is not None and int(row["population"]) == 0
)

population_rows = [
    {
        "kommunenummer": row["Region"].zfill(4),
        "population": int(row["population"]),
        "population_year": int(POPULATION_YEAR),
        "source_table": SSB_TABLE,
        "source_retrieved_at": datetime.now(timezone.utc).isoformat(),
    }
    for row in observations
    if row["population"] is not None and int(row["population"]) > 0
]

assert population_rows, "SSB returned no municipality with a positive population"
assert len({row["kommunenummer"] for row in population_rows}) == len(population_rows)
assert all(len(row["kommunenummer"]) == 4 for row in population_rows)
assert not set(retired_codes) & {row["kommunenummer"] for row in population_rows}

print("Four-digit codes returned by SSB :", len(observations))
print("  retired, dropped               :", len(retired_codes))
print("  in use, kept                   :", len(population_rows))
print("Population range:", min(row["population"] for row in population_rows), "to", max(row["population"] for row in population_rows))
print("Total population:", sum(row["population"] for row in population_rows))

Four-digit codes returned by SSB : 1184
  retired, dropped               : 827
  in use, kept                   : 357
Population range: 219 to 728714
Total population: 5627400


## Write and verify the Parquet dimension

This file is intentionally separate from the existing company and financial mirrors. The analytics build can join it by `kommunenummer` without changing the benchmark inputs.

In [5]:
population_df = spark.createDataFrame(population_rows)

write_staged(population_df, OUTPUT_PATH, "parquet")

written = spark.read.parquet(OUTPUT_PATH)
written_count = written.count()
distinct_count = written.select("kommunenummer").distinct().count()

assert written_count == len(population_rows)
assert distinct_count == written_count
assert written.filter(F.col("population").isNull()).count() == 0
# The retired codes must not survive the round trip either: they are the ones
# that would otherwise join as a population of zero.
assert written.filter(F.col("population") <= 0).count() == 0

written.orderBy("kommunenummer").show(10, truncate=False)
print("Wrote", written_count, "municipality rows to", OUTPUT_PATH)

  staged write: 10 files, 0.00 GB copied to /home/jovyan/data/parquet/ssb_population_2026.parquet


+-------------+----------+---------------+--------------------------------+------------+
|kommunenummer|population|population_year|source_retrieved_at             |source_table|
+-------------+----------+---------------+--------------------------------+------------+
|0301         |728714    |2026           |2026-09-12T07:39:14.498405+00:00|06913       |
|1101         |15546     |2026           |2026-09-12T07:39:14.498578+00:00|06913       |
|1103         |151669    |2026           |2026-09-12T07:39:14.498579+00:00|06913       |
|1106         |38663     |2026           |2026-09-12T07:39:14.498581+00:00|06913       |
|1108         |85785     |2026           |2026-09-12T07:39:14.498582+00:00|06913       |
|1111         |3356      |2026           |2026-09-12T07:39:14.498583+00:00|06913       |
|1112         |3229      |2026           |2026-09-12T07:39:14.498585+00:00|06913       |
|1114         |2894      |2026           |2026-09-12T07:39:14.498586+00:00|06913       |
|1119         |20087 